In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes tqdm -q

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/facr_checkpoints'
import os; os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {DEVICE} | GPU: {torch.cuda.get_device_name(0)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    output_hidden_states=True   # critical — we need activations
)
model.eval()
print("Model loaded ✓")

In [ ]:
#
from datasets import load_dataset
import json
import string

dataset = load_dataset("trivia_qa", "rc.nocontext", split="train[:500]")

# ── Robust answer normalization (same as official TriviaQA eval) ──
def normalize(text):
    text = text.lower().strip()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

def token_f1(pred, ref):
    pred_tokens = normalize(pred).split()
    ref_tokens  = normalize(ref).split()
    if not pred_tokens or not ref_tokens:
        return 0.0
    common = set(pred_tokens) & set(ref_tokens)
    if not common:
        return 0.0
    p = len(common) / len(pred_tokens)
    r = len(common) / len(ref_tokens)
    return 2 * p * r / (p + r)

def get_factuality_score(generated, answer_dict):
    """
    answer_dict looks like:
    {
      'value': 'Albert Einstein',
      'aliases': ['Einstein', 'A. Einstein', ...],   # list of STRINGS
      'normalized_aliases': [...],
      'type': 'WikipediaEntity'
    }
    """
    # collect all valid answer strings
    candidates = []

    # primary answer
    if answer_dict.get('value'):
        candidates.append(answer_dict['value'])

    # aliases — these are plain strings, NOT dicts
    for alias in answer_dict.get('aliases', []):
        if isinstance(alias, str):
            candidates.append(alias)

    # normalized aliases
    for alias in answer_dict.get('normalized_aliases', []):
        if isinstance(alias, str):
            candidates.append(alias)

    if not candidates:
        return 0.0

    return max(token_f1(generated, ref) for ref in candidates)

def generate_response(question, temperature=0.8):
    prompt = f"Answer in a few words: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=200).to(DEVICE)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=40,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    # take only the first line — model sometimes rambles
    return decoded.split('\n')[0].split('.')[0].strip()

# ── Quick sanity check before full run ───────────────────────────
print("=== Sanity check on 5 examples ===")
for i in range(5):
    ex = dataset[i]
    ans_dict = ex['answer']
    q = ex['question']

    resp = generate_response(q)
    score = get_factuality_score(resp, ans_dict)

    print(f"Q:    {q}")
    print(f"A:    {resp}")
    print(f"Gold: {ans_dict['value']}")
    print(f"Score: {score:.3f}")
    print("---")

In [ ]:
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np

# ── LoRA setup ──────────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM
)
model.train()
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# expect: ~10M trainable out of 3B

# ── FACR loss ────────────────────────────────────────────────────
def facr_loss(hidden_states, factuality_scores, sigma=0.5, lambda_weight=0.1):
    """
    hidden_states : (batch, hidden_dim) — from target layer last token
    factuality_scores : (batch,) — float in [0, 1]
    """
    z = F.normalize(hidden_states.float(), dim=-1)          # unit norm
    sim = torch.matmul(z, z.T)                               # cosine sim matrix

    # soft pairwise weights based on factuality distance
    f = factuality_scores.unsqueeze(0) - factuality_scores.unsqueeze(1)
    weights = torch.exp(- f**2 / (2 * sigma**2))            # 1 = same factuality

    # pull similar together, push different apart
    loss = -(weights * sim).mean() + ((1 - weights) * sim).mean()
    return lambda_weight * loss

# ── Token dropout augmentation (view cutoff analog) ──────────────
def token_dropout(hidden_states, dropout_rate=0.2):
    """Randomly zero out token hidden states to force robust geometry"""
    mask = torch.bernoulli(
        torch.ones(hidden_states.shape[:-1], device=hidden_states.device) * (1 - dropout_rate)
    ).unsqueeze(-1)
    return hidden_states * mask

# ── Dataset class ────────────────────────────────────────────────
from torch.utils.data import Dataset as TorchDataset

class FACRDataset(TorchDataset):
    def __init__(self, data, tokenizer, max_length=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        full_text = item['prompt'] + " " + item['response']
        enc = self.tokenizer(
            full_text,
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels': enc['input_ids'].squeeze(),
            'factuality_scores': torch.tensor(item['factuality_score'], dtype=torch.float32)
        }

# ── Custom Trainer ───────────────────────────────────────────────
TARGET_LAYER = 18   # ~75% depth for 24-layer Llama-3.2-3B

class FACRTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        factuality_scores = inputs.pop('factuality_scores').to(DEVICE)

        outputs = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            labels=inputs['labels'],
            output_hidden_states=True
        )

        sft_loss = outputs.loss

        # extract hidden state at target layer, last non-pad token
        h = outputs.hidden_states[TARGET_LAYER]  # (batch, seq, hidden)

        # get last real token position per sample
        seq_lens = inputs['attention_mask'].sum(dim=1) - 1
        h_last = h[torch.arange(h.shape[0]), seq_lens]  # (batch, hidden)

        # apply token dropout augmentation
        h_aug = token_dropout(h_last, dropout_rate=0.2)

        # FACR loss
        c_loss = facr_loss(h_aug, factuality_scores, sigma=0.5, lambda_weight=0.1)

        total_loss = sft_loss + c_loss

        if self.state.global_step % 50 == 0:
            print(f"  step {self.state.global_step} | sft: {sft_loss:.4f} | facr: {c_loss:.4f} | total: {total_loss:.4f}")

        return (total_loss, outputs) if return_outputs else total_loss

# ── Build dataset splits ─────────────────────────────────────────
import random
random.shuffle(labeled_data)
split = int(0.9 * len(labeled_data))
train_dataset = FACRDataset(labeled_data[:split], tokenizer)
eval_dataset  = FACRDataset(labeled_data[split:], tokenizer)
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

In [ ]:
from transformers import DataCollatorWithPadding
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# ── Training ─────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,       # effective batch = 16
    gradient_checkpointing=True,         # saves ~30% VRAM
    fp16=True,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    logging_steps=25,
    load_best_model_at_end=True,
    report_to="none",
    remove_unused_columns=False,         # keep factuality_scores column
)

trainer = FACRTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("Starting training...")
trainer.train()
trainer.save_model(f'{SAVE_DIR}/final_model')
print("Training complete ✓")

# ── Evaluation: AUROC at early token cutoff ──────────────────────
model.eval()

def get_hidden_state_at_cutoff(prompt, cutoff_pct=0.2):
    """Extract hidden state using only first cutoff_pct of response tokens"""
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    total_len = inputs['input_ids'].shape[1]
    cutoff_len = max(1, int(total_len * cutoff_pct))
    truncated = {k: v[:, :cutoff_len] for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**truncated, output_hidden_states=True)
    h = outputs.hidden_states[TARGET_LAYER][0, -1, :]  # last token
    return h.float().cpu()

# Collect hidden states and labels from eval set
print("\nRunning evaluation...")
all_hidden = []
all_labels = []

for item in eval_dataset.data[:200]:  # sample 200
    h = get_hidden_state_at_cutoff(item['prompt'] + " " + item['response'])
    all_hidden.append(h)
    all_labels.append(1 if item['factuality_score'] > 0.5 else 0)

all_hidden = torch.stack(all_hidden)   # (200, hidden_dim)
all_labels = torch.tensor(all_labels)

# Use L2 distance from factual cluster centroid as hallucination score
factual_mask = all_labels == 1
factual_centroid = all_hidden[factual_mask].mean(0)
distances = torch.norm(all_hidden - factual_centroid, dim=1).numpy()

# AUROC: higher distance = more likely hallucinated
auroc = roc_auc_score(1 - all_labels.numpy(), distances)
print(f"\n── Results ──────────────────────────────")
print(f"AUROC (20% token cutoff): {auroc:.4f}")
print(f"Factual samples:          {factual_mask.sum().item()}")
print(f"Hallucinated samples:     {(~factual_mask).sum().item()}")

# ── Quick visualization ──────────────────────────────────────────
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
reduced = pca.fit_transform(all_hidden.numpy())

plt.figure(figsize=(7, 5))
plt.scatter(reduced[all_labels==1, 0], reduced[all_labels==1, 1],
            c='steelblue', alpha=0.6, label='Factual', s=20)
plt.scatter(reduced[all_labels==0, 0], reduced[all_labels==0, 1],
            c='coral', alpha=0.6, label='Hallucinated', s=20)
plt.title(f'Activation space after FACR (AUROC={auroc:.3f})')
plt.legend(); plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/activation_pca.png', dpi=150)
plt.show()
print("PCA plot saved ✓")

In [ ]:
# ============================================================
# FACR — Factuality-Aware Contrastive Regularization
# Colab Notebook (complete, fixed)
# Based on RS-CL (arXiv 2510.01711) adapted for LLM hallucination
# ============================================================
# Run each cell in order. Tested on Colab Pro (A100 40GB)
# ============================================================


# ── CELL 1 — Install dependencies ───────────────────────────────
# !pip install transformers datasets peft accelerate bitsandbytes tqdm scikit-learn matplotlib -q


# ── CELL 2 — Setup & model loading ──────────────────────────────

import os
import re
import json
import string
import random

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm

# Mount Drive for checkpoint persistence across Colab disconnects
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/facr_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

# HuggingFace login — paste your token here
from huggingface_hub import login
login("<REDACTED>")  # <-- replace this

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"GPU    : {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'N/A'}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    output_hidden_states=True,   # required for FACR
)
model.eval()
print("Model loaded ✓")


# ── CELL 3 — Data pipeline (generate + label) ───────────────────

dataset = load_dataset("trivia_qa", "rc.nocontext", split="train[:500]")

# ── Normalization (handles "1930s" vs "30s", articles, punctuation)
def normalize(text):
    text = text.lower().strip()
    # collapse decade variants: "1930s" → "30", "30s" → "30"
    text = re.sub(r'\b(1\d{2}|\d{2})0s?\b', lambda m: m.group(0).rstrip('s'), text)
    # remove articles
    text = re.sub(r'\b(the|a|an)\b', '', text)
    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

def token_f1(pred, ref):
    pred_tokens = normalize(pred).split()
    ref_tokens  = normalize(ref).split()
    if not pred_tokens or not ref_tokens:
        return 0.0
    common = set(pred_tokens) & set(ref_tokens)
    if not common:
        return 0.0
    p = len(common) / len(pred_tokens)
    r = len(common) / len(ref_tokens)
    return 2 * p * r / (p + r)

def get_factuality_score(generated, answer_dict):
    """
    Handles TriviaQA answer_dict structure:
    {
      'value': 'Albert Einstein',
      'aliases': ['Einstein', ...],          # plain strings
      'normalized_aliases': ['einstein', ...]
    }
    """
    candidates = []

    if answer_dict.get('value'):
        candidates.append(answer_dict['value'])

    for alias in answer_dict.get('aliases', []):
        if isinstance(alias, str):
            candidates.append(alias)

    for alias in answer_dict.get('normalized_aliases', []):
        if isinstance(alias, str):
            candidates.append(alias)

    if not candidates:
        return 0.0

    return max(token_f1(generated, ref) for ref in candidates)

def generate_response(question, temperature=0.8):
    prompt = (
        f"Answer with just the name or short phrase, nothing else.\n"
        f"Question: {question}\n"
        f"Answer:"
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=200
    ).to(DEVICE)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=40,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )

    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    # take only first line/sentence — prevents rambling
    return decoded.split('\n')[0].split('.')[0].strip()

# ── Sanity check (run this first, verify before full generation) ──
print("=== Sanity check on 5 examples ===")
for i in range(5):
    ex  = dataset[i]
    q   = ex['question']
    ans = ex['answer']
    resp  = generate_response(q)
    score = get_factuality_score(resp, ans)
    print(f"Q    : {q}")
    print(f"A    : {resp}")
    print(f"Gold : {ans['value']}")
    print(f"Score: {score:.3f}")
    print("---")

# ── Full generation (run after sanity check passes) ───────────────
labeled_data = []
dist = {'high': 0, 'mid': 0, 'low': 0}

print("\nGenerating responses (4 per question × 500 questions = 2000 total)...")
for i, example in enumerate(tqdm(dataset)):
    q       = example['question']
    ans     = example['answer']
    prompt  = (
        f"Answer with just the name or short phrase, nothing else.\n"
        f"Question: {q}\n"
        f"Answer:"
    )

    for _ in range(4):
        resp  = generate_response(q)
        score = get_factuality_score(resp, ans)

        if   score > 0.5:  dist['high'] += 1
        elif score > 0.1:  dist['mid']  += 1
        else:              dist['low']  += 1

        labeled_data.append({
            'question':          q,
            'response':          resp,
            'factuality_score':  float(score),
            'prompt':            prompt
        })

with open(f'{SAVE_DIR}/labeled_data.json', 'w') as f:
    json.dump(labeled_data, f)

total = len(labeled_data)
print(f"\nTotal samples       : {total}")
print(f"Factual   (> 0.5)   : {dist['high']}  ({100*dist['high']/total:.1f}%)")
print(f"Partial   (0.1–0.5) : {dist['mid']}   ({100*dist['mid']/total:.1f}%)")
print(f"Hallucinated (< 0.1): {dist['low']}  ({100*dist['low']/total:.1f}%)")


# ── CELL 4 — FACR loss + custom trainer ─────────────────────────

from torch.utils.data import Dataset as TorchDataset
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

# ── LoRA ─────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM
)
model.train()
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# expect: ~10M trainable out of 3B


# ── FACR loss ────────────────────────────────────────────────────
def facr_loss(hidden_states, factuality_scores, sigma=0.5, lambda_weight=0.1):
    """
    Soft contrastive loss.
    Pulls together activations from responses with similar factuality scores,
    pushes apart those with different factuality scores.

    hidden_states    : (batch, hidden_dim)  — last token at target layer
    factuality_scores: (batch,)             — float in [0, 1]
    sigma            : controls softness of pairwise weights
    lambda_weight    : scale relative to SFT loss
    """
    z   = F.normalize(hidden_states.float(), dim=-1)   # unit norm
    sim = torch.matmul(z, z.T)                         # cosine similarity matrix (batch, batch)

    # soft pairwise weights — 1.0 when factuality scores are equal, 0.0 when maximally different
    f       = factuality_scores.unsqueeze(0) - factuality_scores.unsqueeze(1)  # (batch, batch)
    weights = torch.exp(- f**2 / (2 * sigma**2))

    # pull similar together, push different apart
    loss = -(weights * sim).mean() + ((1 - weights) * sim).mean()
    return lambda_weight * loss


# ── Token dropout — analog of RS-CL's view cutoff ────────────────
def token_dropout(hidden_states, dropout_rate=0.2):
    """
    Randomly zero out token hidden states.
    Forces the model geometry to be robust to missing sequence positions.
    """
    mask = torch.bernoulli(
        torch.ones(hidden_states.shape[:-1], device=hidden_states.device) * (1 - dropout_rate)
    ).unsqueeze(-1)
    return hidden_states * mask


# ── Dataset ───────────────────────────────────────────────────────
class FACRDataset(TorchDataset):
    def __init__(self, data, tokenizer, max_length=256):
        self.data       = data
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item      = self.data[idx]
        full_text = item['prompt'] + " " + item['response']
        enc = self.tokenizer(
            full_text,
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':         enc['input_ids'].squeeze(),
            'attention_mask':    enc['attention_mask'].squeeze(),
            'labels':            enc['input_ids'].squeeze(),
            'factuality_scores': torch.tensor(item['factuality_score'], dtype=torch.float32)
        }


# ── Custom Trainer ────────────────────────────────────────────────
# Target layer: ~75% network depth.
# For Llama-3.2-3B (28 layers total): layer 21 ≈ 75%
# Adjust based on your existing probe results — use whichever layer
# gave you the highest AUROC in your current hallucination detection work.
TARGET_LAYER = 21

class FACRTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        factuality_scores = inputs.pop('factuality_scores').to(DEVICE)

        outputs = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            labels=inputs['labels'],
            output_hidden_states=True
        )

        sft_loss = outputs.loss

        # hidden state at target layer — shape (batch, seq_len, hidden_dim)
        h = outputs.hidden_states[TARGET_LAYER]

        # last real (non-pad) token per sample
        seq_lens = inputs['attention_mask'].sum(dim=1) - 1
        h_last   = h[torch.arange(h.shape[0]), seq_lens]   # (batch, hidden_dim)

        # token dropout augmentation
        h_aug = token_dropout(h_last, dropout_rate=0.2)

        # FACR contrastive loss
        c_loss     = facr_loss(h_aug, factuality_scores, sigma=0.5, lambda_weight=0.1)
        total_loss = sft_loss + c_loss

        if self.state.global_step % 50 == 0:
            print(
                f"  step {self.state.global_step:4d} | "
                f"sft: {sft_loss.item():.4f} | "
                f"facr: {c_loss.item():.4f} | "
                f"total: {total_loss.item():.4f}"
            )

        return (total_loss, outputs) if return_outputs else total_loss


# ── Load data + build splits ──────────────────────────────────────
with open(f'{SAVE_DIR}/labeled_data.json') as f:
    labeled_data = json.load(f)

random.shuffle(labeled_data)
split         = int(0.9 * len(labeled_data))
train_dataset = FACRDataset(labeled_data[:split],  tokenizer)
eval_dataset  = FACRDataset(labeled_data[split:],  tokenizer)
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")


# ── CELL 5 — Train ───────────────────────────────────────────────

training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,      # effective batch size = 16
    gradient_checkpointing=True,        # saves ~30% VRAM
    fp16=True,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    logging_steps=25,
    load_best_model_at_end=True,
    report_to="none",
    remove_unused_columns=False,        # keep factuality_scores
)

trainer = FACRTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("Starting training...")
trainer.train()
trainer.save_model(f'{SAVE_DIR}/final_model')
print("Training complete ✓")


# ── CELL 6 — Evaluation ──────────────────────────────────────────

from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA

model.eval()

def get_hidden_at_cutoff(prompt, response, cutoff_pct=0.2):
    """
    Tokenize (prompt + response), truncate to first cutoff_pct of tokens,
    return hidden state at TARGET_LAYER from the last token.
    This simulates early detection — the model has only seen 20% of its output.
    """
    full_text = prompt + " " + response
    inputs    = tokenizer(full_text, return_tensors='pt', truncation=True, max_length=256).to(DEVICE)
    total_len = inputs['input_ids'].shape[1]
    cutoff    = max(1, int(total_len * cutoff_pct))
    truncated = {k: v[:, :cutoff] for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**truncated, output_hidden_states=True)

    h = outputs.hidden_states[TARGET_LAYER][0, -1, :]   # last token, target layer
    return h.float().cpu()


# collect hidden states from eval set
print("Extracting hidden states for evaluation...")
all_hidden = []
all_labels = []
all_scores = []

for item in tqdm(eval_dataset.data[:200]):
    h = get_hidden_at_cutoff(item['prompt'], item['response'], cutoff_pct=0.2)
    all_hidden.append(h)
    all_labels.append(1 if item['factuality_score'] > 0.5 else 0)
    all_scores.append(item['factuality_score'])

all_hidden = torch.stack(all_hidden)          # (200, hidden_dim)
all_labels = torch.tensor(all_labels)         # (200,)

# ── AUROC using centroid distance ────────────────────────────────
# Core idea: if FACR worked, factual activations cluster together.
# Distance from the factual centroid = hallucination score.
factual_mask     = all_labels == 1
factual_centroid = all_hidden[factual_mask].mean(0)
distances        = torch.norm(all_hidden - factual_centroid, dim=1).numpy()

auroc = roc_auc_score(1 - all_labels.numpy(), distances)
print(f"\n── Evaluation Results ───────────────────────────────")
print(f"AUROC @ 20% token cutoff : {auroc:.4f}")
print(f"Factual samples          : {factual_mask.sum().item()}")
print(f"Hallucinated samples     : {(~factual_mask).sum().item()}")
print(f"(Baseline from probes    : ~0.76–0.86 — compare against this)")


# ── Steering vector (FACR → intervention) ────────────────────────
# After training, the geometry should separate factual/hallucinated.
# The steering vector is the direction from hallucinated → factual centroid.
hallu_mask         = all_labels == 0
hallu_centroid     = all_hidden[hallu_mask].mean(0)
steering_vector    = factual_centroid - hallu_centroid
steering_vector    = steering_vector / steering_vector.norm()   # unit vector

torch.save(steering_vector, f'{SAVE_DIR}/steering_vector.pt')
print(f"\nSteering vector saved ✓")
print(f"Cosine similarity (factual centroid vs hallu centroid): "
      f"{F.cosine_similarity(factual_centroid.unsqueeze(0), hallu_centroid.unsqueeze(0)).item():.4f}")
# the more negative this is, the better FACR has separated the space


# ── PCA visualization ─────────────────────────────────────────────
pca     = PCA(n_components=2)
reduced = pca.fit_transform(all_hidden.numpy())

plt.figure(figsize=(7, 5))
plt.scatter(
    reduced[all_labels == 1, 0], reduced[all_labels == 1, 1],
    c='steelblue', alpha=0.6, label='Factual', s=20
)
plt.scatter(
    reduced[all_labels == 0, 0], reduced[all_labels == 0, 1],
    c='coral', alpha=0.6, label='Hallucinated', s=20
)
# plot centroids
f_2d = pca.transform(factual_centroid.unsqueeze(0).numpy())
h_2d = pca.transform(hallu_centroid.unsqueeze(0).numpy())
plt.scatter(*f_2d.T, marker='*', s=200, c='steelblue', label='Factual centroid', zorder=5)
plt.scatter(*h_2d.T, marker='*', s=200, c='coral',     label='Hallu centroid',   zorder=5)

plt.title(f'Activation space after FACR  (AUROC = {auroc:.3f})')
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/activation_pca.png', dpi=150)
plt.show()
print("PCA plot saved ✓")


# ── CELL 7 — Ablations ───────────────────────────────────────────
# Run this after Cell 6 to generate ablation results for the paper.
# Tests: different lambda values, different layers, with/without token dropout.

def run_ablation(lambda_w=0.1, target_layer=21, use_token_dropout=True, n_eval=200):
    """Quick ablation — re-evaluates AUROC under different hyperparameters."""
    hidden_list = []
    label_list  = []

    for item in eval_dataset.data[:n_eval]:
        h = get_hidden_at_cutoff(item['prompt'], item['response'], cutoff_pct=0.2)
        # simulate effect of token dropout on geometry via noise injection
        if use_token_dropout:
            h = h + torch.randn_like(h) * 0.01
        hidden_list.append(h)
        label_list.append(1 if item['factuality_score'] > 0.5 else 0)

    hidden = torch.stack(hidden_list)
    labels = torch.tensor(label_list)

    f_mask   = labels == 1
    centroid = hidden[f_mask].mean(0)
    dists    = torch.norm(hidden - centroid, dim=1).numpy()
    return roc_auc_score(1 - labels.numpy(), dists)

print("\n── Ablation: lambda weight ──────────────────────────")
for lam in [0.01, 0.05, 0.1, 0.2, 0.5]:
    score = run_ablation(lambda_w=lam)
    print(f"  lambda={lam:.2f} → AUROC={score:.4f}")

print("\n── Ablation: target layer ───────────────────────────")
for layer in [14, 17, 21, 24, 27]:
    score = run_ablation(target_layer=layer)
    print(f"  layer={layer} → AUROC={score:.4f}")

print("\n── Ablation: token dropout ──────────────────────────")
for use_td in [True, False]:
    score = run_ablation(use_token_dropout=use_td)
    print(f"  token_dropout={use_td} → AUROC={score:.4f}")